# Push/Fold RL — Fase 4 do plano (rodando local, kernel `tf_env` com CUDA)

Treina um agente PPO no ambiente `PushFoldOpenEnv`/`PushFoldFacingShoveEnv` (`poker_coach/poker_coach/rl/pushfold_env.py`), reaproveitando o motor Nash já existente (`pushfold/nash.py`) como reward, vilão e referência de avaliação.

**Versão local (não Colab):** roda direto no repo já clonado na máquina, sem `git clone`, sem upload/download de arquivo — o dataset e o artefato final ficam em disco, nos caminhos já usados pelo resto do projeto.

**Honestidade sobre GPU (RTX 4060/CUDA local ou qualquer outra):** esse ambiente específico tem observação de 3 dimensões e 2 ações — a política é uma rede de ~2 camadas de 32 neurônios, minúscula. O Stable-Baselines3 avisa (e o smoke-test já confirmou nesta sessão) que PPO com `MlpPolicy` desse tamanho não ganha nada de GPU — a sobrecarga de transferência CPU↔GPU supera o ganho, então é normal a GPU ficar praticamente ociosa aqui. GPU importa de verdade quando o ambiente crescer (pós-flop, roadmap V6+). Rodar aqui local com CUDA disponível não atrapalha nem exige nada especial — só não espere uso de GPU visível.

**Cuidado com leakage do benchmark:** reward, vilão e avaliação usam o MESMO motor Nash. Isso mede se a infraestrutura de RL converge pra uma aproximação do equilíbrio chip-EV já conhecido — não é uma alegação de estratégia superior a Nash.

**Se der erro de import no meio do caminho:** reinicie o kernel (Kernel → Restart) antes de rodar de novo do zero. `sys.modules` não se limpa sozinho quando um import quebra no meio — rodar as células de novo sem reiniciar reaproveita o módulo quebrado que ficou preso lá (a célula 2b já tenta se proteger disso, mas reiniciar o kernel é a garantia real).

## 1. Instalar dependências (se o kernel `tf_env` ainda não tiver)

In [28]:
import sys

# `!pip` roda o pip do PATH do shell, que pode não ser o do kernel
# `tf_env` (comum quando existe mais de um ambiente conda/venv na
# máquina) — `{sys.executable} -m pip` garante instalar no MESMO
# interpretador Python que está rodando este notebook.
!{sys.executable} -m pip install -q stable-baselines3 gymnasium pandas pyarrow

/bin/bash: /home/felipe/miniconda3/envs/tf_env/lib/libtinfo.so.6: no version information available (required by /bin/bash)


## 2. Importar o pacote `poker_coach` (local — repo já está no disco, sem clone)

In [30]:
import sys

REPO_ROOT = "/home/felipe/Projeto/poker_coach"
sys.path.insert(0, f"{REPO_ROOT}/poker_coach")

## 2b. Compatibilidade com Python 3.8 (kernel `tf_env`)

O pacote `poker_coach` roda em produção com Python 3.12 (API/app principal) e várias partes dele usam anotações modernas (`list[X]`, `dict[K, V]`, `X | None`) sem `from __future__ import annotations` no arquivo — em Python 3.8 essas anotações são avaliadas na hora que a função/classe é definida (não só quando é chamada), e quebram na hora (achado real: não é só `pushfold_env.py`, `models.py` também quebra do mesmo jeito assim que é importado por baixo — teria sido caçar arquivo por arquivo).

**Não mexe nos arquivos do repo** (a app principal em 3.12 não precisa disso) — instala um *import hook* nesta célula que intercepta QUALQUER `import poker_coach...` pelo resto do notebook e compila com a mesma flag que `from __future__ import annotations` liga (adia toda anotação pra string, sem executar `list[X]`/`X | None` de verdade). Cobre o pacote inteiro de uma vez, não só os arquivos que já bateram erro. O único caso que a flag sozinha não resolve é `handeval.py::Card = tuple[int, str]` — isso é uma ATRIBUIÇÃO em runtime, não anotação — então o hook troca só essa linha por `typing.Tuple[int, str]` (equivalente, funciona em qualquer versão) só nessa cópia carregada em memória.

A célula também limpa `sys.modules` de qualquer `poker_coach.*` que já tenha ficado preso quebrado de uma tentativa anterior (rodar essa célula de novo, mesmo sem reiniciar o kernel, se auto-corrige).

In [31]:
import __future__
import importlib.abc
import importlib.machinery

# Limpa qualquer `poker_coach`/`poker_coach.*` que já tenha sido
# importado QUEBRADO numa tentativa anterior nesta mesma sessão do
# kernel — se essa célula já rodou antes e um import no meio deu erro,
# o módulo parcialmente executado fica preso em `sys.modules` (Python só
# limpa isso sozinho quando é o MECANISMO padrão de import que falha,
# não quando é um `exec()` manual dentro de um loader customizado). Sem
# isso, rodar essa célula de novo sem reiniciar o kernel reusaria o
# módulo quebrado de antes.
for _name in [m for m in list(sys.modules) if m == "poker_coach" or m.startswith("poker_coach.")]:
    del sys.modules[_name]


class _Py38AnnotationsLoader(importlib.machinery.SourceFileLoader):
    """Compila qualquer módulo do poker_coach com a flag de `from
    __future__ import annotations` ligada, mesmo que o arquivo não tenha
    essa linha — sem editar o arquivo de verdade no repo."""

    def source_to_code(self, data, path, *, _optimize=-1):
        src = data.decode("utf-8") if isinstance(data, (bytes, bytearray)) else data
        if path.endswith("handeval.py"):
            # unico caso de ATRIBUICAO em runtime (nao anotacao) — a flag
            # nao ajuda aqui, so troca de texto mesmo resolve.
            src = src.replace(
                "Card = tuple[int, str]",
                "from typing import Tuple as _Tuple38\nCard = _Tuple38[int, str]",
            )
        return compile(
            src, path, "exec", dont_inherit=True,
            flags=__future__.annotations.compiler_flag,
        )

    def get_code(self, fullname):
        # Raiz do erro que persistiu mesmo com o hook instalado: o
        # `SourceFileLoader.get_code` padrão (que a gente herda e NÃO
        # sobrescrevia até agora) procura primeiro um .pyc em cache
        # (`__pycache__/*.cpython-38.pyc`) e, se o .py fonte não mudou,
        # usa esse bytecode direto — nunca chama `source_to_code` de
        # novo. Se `models.py`/`pushfold_env.py` etc já tinham sido
        # importados ALGUMA vez neste venv sem o hook (ex.: tentativa
        # anterior desta mesma sessão), o .pyc cacheado não tem a flag
        # de annotations adiada, e o TypeError do `list[Seat]` volta
        # mesmo com o hook instalado e com o `sys.modules` limpo. Aqui a
        # gente ignora esse cache por completo — compila do source
        # sempre, sem ler nem escrever .pyc (pacote pequeno, custo
        # irrelevante).
        path = self.get_filename(fullname)
        source_bytes = self.get_data(path)
        return self.source_to_code(source_bytes, path)


class _Py38CompatFinder(importlib.abc.MetaPathFinder):
    """Intercepta qualquer import `poker_coach` ou `poker_coach.*` —
    acha o arquivo do jeito padrão (PathFinder), só troca o loader."""

    def find_spec(self, fullname, path, target=None):
        if fullname != "poker_coach" and not fullname.startswith("poker_coach."):
            return None
        spec = importlib.machinery.PathFinder.find_spec(fullname, path)
        if spec is None or spec.origin is None or not spec.origin.endswith(".py"):
            return None
        spec.loader = _Py38AnnotationsLoader(fullname, spec.origin)
        spec.cached = None
        return spec


# Remove qualquer instância antiga do hook antes de inserir (rodar essa
# célula 2x seguidas não empilha finders duplicados).
sys.meta_path = [f for f in sys.meta_path if not isinstance(f, _Py38CompatFinder)]
sys.meta_path.insert(0, _Py38CompatFinder())
print("Hook de compatibilidade Python 3.8 instalado — vale pro resto do notebook.")

Hook de compatibilidade Python 3.8 instalado — vale pro resto do notebook.


## 3. Checar GPU/CUDA visível pro kernel

In [32]:
import torch

print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
# Esperado: SB3 vai preferir/usar CPU de qualquer forma pra essa rede
# minúscula (ver nota da célula 1) — não é bug se o uso de GPU ficar ~0%.

CUDA disponível: True
GPU: NVIDIA GeForce RTX 4060


## 4. Carregar o dataset (local, sem upload)

Gerado via `python -m poker_coach.cli export-rl --out poker_coach_decisions.parquet` (Fase 1). Usado aqui só pra CALIBRAR a distribuição de `(effective_bb, pot_bb)` dos episódios de treino com spots reais — o reward continua vindo do Nash, não do resultado real da mão (ver `export.py`: outcome != qualidade de decisão).

In [33]:
import pandas as pd

from poker_coach.rl.pushfold_env import (
    PushFoldFacingShoveEnv,
    PushFoldOpenEnv,
    evaluate_against_nash,
    sample_states_from_dataset,
)

dataset_path = f"{REPO_ROOT}/datasets/poker_coach_decisions.parquet"
df = pd.read_parquet(dataset_path)
open_states = sample_states_from_dataset(df, "preflop_open")
facing_states = sample_states_from_dataset(df, "preflop_facing_allin") + sample_states_from_dataset(
    df, "preflop_multiway_facing_allin"
)
print(f"Spots calibrados: {len(open_states)} de abertura, {len(facing_states)} facing-shove")

Spots calibrados: 7373 de abertura, 962 facing-shove


## 5. Treinar PPO — abertura preflop (mais timesteps do que o smoke-test local)

O smoke-test com 5.000 timesteps já levou a concordância de 50% (baseline aleatório) pra ~88% em segundos. Aqui usamos um budget bem maior pra ver onde converge de verdade.

In [34]:
from poker_coach.rl.train_pushfold_ppo import agent_predict_fn, format_training_report, save_model, train_and_benchmark

report, model = train_and_benchmark(
    total_timesteps=300_000,
    seed=0,
    state_samples=open_states,
    grid_points=25,
    env_cls=PushFoldOpenEnv,
)
print(format_training_report(report))

KeyboardInterrupt: 

## 6. Avaliação — as 3 métricas (agreement / EV gap / generalização)

`train_and_benchmark` já roda A) agreement e B) EV gap contra uma grade held-out (C, generalização — nunca usada como episódio de treino). Célula abaixo roda o mesmo agente contra grades com seeds diferentes pra checar estabilidade.

In [ ]:
predict = agent_predict_fn(model)
for seed in (1, 2, 3):
    r = evaluate_against_nash(PushFoldOpenEnv, predict, grid_points=25, seed=seed)
    print(
        f"seed={seed}: agreement={r.agreement_pct}%  mean_ev_gap={r.mean_ev_gap}  "
        f"clear={r.clear_agreement_pct}%(n={r.clear_n})  borderline={r.borderline_agreement_pct}%(n={r.borderline_n})"
    )

## 7. (Opcional) Facing-Shove Env — mesmo processo, outro lado do equilíbrio

Usa `env_cls=PushFoldFacingShoveEnv` — `train_and_benchmark`/`train_ppo` aceitam trocar o ambiente (parâmetro adicionado nesta sessão; antes disso a função só treinava `PushFoldOpenEnv`, o comentário antigo do notebook estava incorreto).

In [ ]:
report_facing, model_facing = train_and_benchmark(
    total_timesteps=300_000,
    seed=0,
    state_samples=facing_states,
    grid_points=25,
    env_cls=PushFoldFacingShoveEnv,
)
print(format_training_report(report_facing))

## 8. Exportar os artefatos treinados (local, sem download de navegador)

In [ ]:
save_model(model, f"{REPO_ROOT}/datasets/pushfold_open_ppo.zip")
save_model(model_facing, f"{REPO_ROOT}/datasets/pushfold_facing_shove_ppo.zip")
print("Salvos em", f"{REPO_ROOT}/datasets/")

## Lacuna conhecida (não escondida)

Este notebook treina o PPO **do zero** (init padrão do SB3), não a partir
da Personal Policy (Fase 2, Behavioral Cloning) como pedido originalmente.
Fazer o BC warm-start direito exige transplantar pesos pré-treinados pra
dentro de `model.policy.mlp_extractor`/`action_net`, ou usar a lib
`imitation` (BC + integração nativa com SB3). Ver docstring de
`poker_coach/poker_coach/rl/train_pushfold_ppo.py` pro caminho concreto
de como fechar essa lacuna numa próxima sessão.